<a href="https://colab.research.google.com/github/bholagautam99/git-github-basics/blob/main/Bhola_Gautam_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Chemical Applications of Machine Learning (CHEM 4930/5610) - Spring 2026

### Assignment 3 - Deadline 2/27/2026
Points 10

#### General Comments
All figures and graph should have approriate labels on the two axis, and should include a legend with appropriate labels of the different plots.

The notebook should be return in working format. That is, I should be able to reset all the output and re-run all the cells and get the same results as you obtained.

**You should start by saving a copy of the notebook to your Google Drive so you preserve all changes.**

**Please add your name as a suffix to the filname**

**Student Name**: Bhola Gautam

1.   List item
2.   List item



**AI usage statement:**
I used ChatGPT, Claude, GeminiAI to assist with coding by clarifying concepts, troubleshooting errors, and generating example code snippets. I reviewed, modified, and tested all code myself to ensure it meets the assignment requirements. I take full responsibility for the final submission.

### Task 1 - 10 points

In this task, we will consider data from this paper:
- Enhancing Permeability Prediction of Heterobifunctional Degraders Using Machine Learning and Metadynamics-Informed 3D Molecular Descriptors - [DOI:10.1021/acs.jcim.5c01600](https://doi.org/10.1021/acs.jcim.5c01600)

Where the authors consider the Permeability of so-called PROTAC compounds that are large and flexible molecules used in Targeted Protein Degradation.

All the dataset used in the paper, and the code use to obtain the results are given in this following Github repository:
- https://github.com/brykimjh/degrader-permeability-ml3d-metaD  

The specfic dataset that we use 32 PROTAC compounds with measured passive permeability (given in nm/s) and includes 17 2D features calculated by RDKit (see [here](https://github.com/brykimjh/degrader-permeability-ml3d-metaD/blob/main/data/calculate_2d_properties.py) for the script they are calculated), and 3 3D "ensemble" features that are obtained from molecular dynamics simulations as described in the paper.

The target value is the measured passive permeability that is experimentaly measured and given in nm/s.

The dataset can be seen here:
- https://github.com/brykimjh/degrader-permeability-ml3d-metaD/blob/main/outputs/ml_models/model_data.csv

Where the log10 transformed passive permeability is given by `P_appLog`. Note that here the passive permeability has been log transformed in based 10 using `np.log10`.

The 2D features obtained using RDKit are:
```
[
 'Molecular Weight (MW)', 'CharVol (characteristic volume)',
 'Flexibility (number of rotatable bonds / number of bonds)',
 'Number of Heavy Atoms (HA)', 'RingAtoms', 'Halogens', 'HeteroAtoms',
 'RotBonds (NRotB)', 'AllBonds', 'RingCount', 'NumStereo',
 'Fraction of sp3 Carbon Atoms (FSP3)', 'Hydrogen Bond Donors (HBD)',
 'Hydrogen Bond Acceptors (HBA)', 'cLogD^7.4',
 'Topological polar surface area (TPSA)',
 'Total non-polar surface area (TNSA)'
]
```
that includes various standard descriptors/features implemented in RDKit

and the 3D "ensemble" features obtained using MD simulations are
```
[
 'Ensemble_Average_PSA_Chloroform_ANI',
 'Ensemble_Average_Num_IMHB_Chloroform_ANI',
 'Ensemble_Average_RadiusOfGyration_Chloroform_ANI'
]
```
which inludes the average polar surface area (PSA), the average number of intramolecular hydrogen bonds (IMHB), and the average radius of gyration (that measures if the molecule is compact or extended).

#### A)
Split the dataset into compounds with two classes with low and high permeability evenly so each class has 16 compounds.

#### B)
For all 32 compounds, visualize the 2D chemical structure using RDKit and the mols2grid package, and show their measured passive permeability in nm/s and their low/high permeability classifciation.

#### C)
Perform classifcation using random forest classification model using the following hyperparameters:
- Number of tree: 100
- Maximum depth of each tree: 2
- Maximum number of features for each tree: 50% of the number of input features (rounded up if that number is a fraction)

You should perform the classification using 3 different feature set:
- 2D RDKit features
- 3D Ensemble features
- Combined set of 2D and 3D features

For each case, perform cross validation (CV) using a CV strategy of your choice, and obtain the average and standard deviation of metrics that measure the performance, including:
- Accuracy
- Precision
- Receiver Operating Characteristic Area Under the Curve (ROC AUC)  

In your analysis and metrics, the high permeability class should be consider as the postive class.

Based on your analysis, which feature set gives the best results for the classifcation?

#### D) - Optional for 2 points
Repeat your analysis from C) using a $k$ nearest neighbors classifer (i.e., $k$ nearest neighbors vote). Use the default value of 5 neighbors.

Do you obtain the same results as in C)?



In [221]:
# Bash script to download all the dataset. Don't worry if you don't understand it
%%bash

url="https://raw.githubusercontent.com/brykimjh/degrader-permeability-ml3d-metaD/refs/heads/main/outputs/ml_models/"
dataset_filename="model_data.csv"

rm -f ${dataset_filename}

wget ${url}/${dataset_filename} &> /dev/null

ls

2d_features.csv
2d_features.csv.1
2d_features.csv.2
2d_features.csv.3
2d_features.csv.4
2d_features.csv.5
2d_features.csv.6
bin
model_data.csv
sample_data


In [222]:
# Bash script to download all the dataset. Don't worry if you don't understand it
%%bash

url="https://raw.githubusercontent.com/brykimjh/degrader-permeability-ml3d-metaD/refs/heads/main/outputs/ml_models/"
dataset_filename="model_data.csv"

rm -f ${dataset_filename}

wget ${url}/${dataset_filename} &> /dev/null

ls

2d_features.csv
2d_features.csv.1
2d_features.csv.2
2d_features.csv.3
2d_features.csv.4
2d_features.csv.5
2d_features.csv.6
bin
model_data.csv
sample_data


In [223]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [224]:
dataset_1 = pd.read_csv("model_data.csv")
dataset_2 = pd.read_csv("2d_features.csv")

In [225]:
# Download dataset

%%bash
dataset_url="https://raw.githubusercontent.com/brykimjh/degrader-permeability-ml3d-metaD/refs/heads/main/data/2d_features.csv"
wget ${dataset_url}
ls

2d_features.csv
2d_features.csv.1
2d_features.csv.2
2d_features.csv.3
2d_features.csv.4
2d_features.csv.5
2d_features.csv.6
2d_features.csv.7
bin
model_data.csv
sample_data


--2026-03-03 05:54:30--  https://raw.githubusercontent.com/brykimjh/degrader-permeability-ml3d-metaD/refs/heads/main/data/2d_features.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10653 (10K) [text/plain]
Saving to: ‘2d_features.csv.7’

     0K ..........                                            100% 41.7M=0s

2026-03-03 05:54:30 (41.7 MB/s) - ‘2d_features.csv.7’ saved [10653/10653]



In [226]:
import os
print(os.listdir("sample_data"))

['README.md', 'anscombe.json', 'california_housing_test.csv', 'mnist_train_small.csv', 'mnist_test.csv', 'california_housing_train.csv']


In [227]:
# the %%capture command will surpress output to screen
%%capture
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install rdkit mols2grid

In [228]:
import mols2grid

In [229]:
dataset_2

,Index,Compound,Smiles,P_app AB (nm/s),P_app BA (nm/s),P_app,Molecular Weight (MW),CharVol (characteristic volume),Flexibility (number of rotatable bonds / number of bonds),Number of Heavy Atoms (HA),...,RotBonds (NRotB),AllBonds,RingCount,NumStereo,Fraction of sp3 Carbon Atoms (FSP3),Hydrogen Bond Donors (HBD),Hydrogen Bond Acceptors (HBA),cLogD^7.4,Topological polar surface area (TPSA),Total non-polar surface area (TNSA)
0,1,6a,O=C(C(N1C(C2=CC=CC(NCCOCCOCCOCCC(N(CC3)CCN3C(C...,2.6,370,31.016125,896.999,803.360,0.253521,65.0,...,18.0,71.0,7.0,2.0,0.456522,3.0,14.0,2.64400,209.98,710.02
1,2,6b,O=C(C(N1C(C2=CC=CC(NC(COCCOCC(N(CC3)CCN3C(C=C4...,1.3,96,11.171392,852.902,743.496,0.191176,62.0,...,13.0,68.0,7.0,2.0,0.395349,3.0,13.0,1.76390,217.82,642.18
2,3,6c,O=C(C(N1C(C2=CC=CC(OCC(NCCCCC(N(CC3)CCN3C(C=C4...,1.0,42,6.480741,850.930,753.592,0.191176,62.0,...,13.0,68.0,7.0,2.0,0.409091,3.0,12.0,2.45750,208.59,671.41
3,4,2d,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...,3.5,2,2.900000,1020.210,896.016,0.276316,71.0,...,21.0,76.0,6.0,3.0,0.470588,3.0,12.0,7.70720,186.66,833.34
4,5,4a,CC1=NC(NC2=NC=C(C(=O)NC3=C(C)C=CC=C3Cl)S2)=CC(...,0.2,1,0.400000,1012.704,900.176,0.236842,70.0,...,18.0,76.0,7.0,3.0,0.460000,5.0,14.0,7.42946,214.98,785.02
5,6,2f,CC1(C)C(=O)N(C2=CC=C(C#N)C(C(F)(F)F)=C2)C(=S)N...,17.0,141,49.000000,837.830,697.696,0.265625,59.0,...,17.0,64.0,6.0,1.0,0.375000,1.0,12.0,4.40258,177.04,622.96
6,7,2b,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...,13.5,14,13.700000,1006.183,872.816,0.266667,70.0,...,20.0,75.0,6.0,3.0,0.460000,3.0,12.0,7.31710,186.66,813.34
7,8,2c,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...,2.6,4,3.200000,1128.150,931.232,0.256098,77.0,...,21.0,82.0,6.0,3.0,0.470588,3.0,12.0,8.44280,186.66,833.34
8,9,2a,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...,3.4,86,17.200000,1022.182,891.056,0.276316,71.0,...,21.0,76.0,6.0,3.0,0.460000,3.0,13.0,6.55350,195.89,804.11
9,10,2e,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...,0.6,3,1.300000,1078.143,902.096,0.253165,74.0,...,20.0,79.0,6.0,3.0,0.460000,3.0,12.0,7.80750,186.66,813.34


In [230]:
dataset1

,Molecular Weight (MW),CharVol (characteristic volume),Flexibility (number of rotatable bonds / number of bonds),Number of Heavy Atoms (HA),RingAtoms,Halogens,HeteroAtoms,RotBonds (NRotB),AllBonds,RingCount,...,Fraction of sp3 Carbon Atoms (FSP3),Hydrogen Bond Donors (HBD),Hydrogen Bond Acceptors (HBA),cLogD^7.4,Topological polar surface area (TPSA),Total non-polar surface area (TNSA),Ensemble_Average_PSA_Chloroform_ANI,Ensemble_Average_Num_IMHB_Chloroform_ANI,Ensemble_Average_RadiusOfGyration_Chloroform_ANI,P_appLog
0,896.999,803.360,0.253521,65,42,0,19,18,71,7,...,0.456522,3,14,2.64400,209.98,710.02,193.71,0.97,5.68,1.491362
1,852.902,743.496,0.191176,62,42,0,19,13,68,7,...,0.395349,3,13,1.76390,217.82,642.18,252.14,0.87,5.24,1.049218
2,850.930,753.592,0.191176,62,42,0,18,13,68,7,...,0.409091,3,12,2.45750,208.59,671.41,248.01,0.67,5.47,0.812913
3,1020.210,896.016,0.276316,71,33,3,20,21,76,6,...,0.470588,3,12,7.70720,186.66,833.34,186.01,0.60,5.84,0.462398
4,1012.704,900.176,0.236842,70,39,1,20,18,76,7,...,0.460000,5,14,7.42946,214.98,785.02,105.63,2.92,5.60,-0.397940
5,837.830,697.696,0.265625,59,34,3,19,17,64,6,...,0.375000,1,12,4.40258,177.04,622.96,204.17,0.29,5.23,1.690196
6,1006.183,872.816,0.266667,70,33,3,20,20,75,6,...,0.460000,3,12,7.31710,186.66,813.34,175.16,0.32,5.41,1.136721
7,1128.150,931.232,0.256098,77,33,9,26,21,82,6,...,0.470588,3,12,8.44280,186.66,833.34,160.07,0.93,5.93,0.505150
8,1022.182,891.056,0.276316,71,33,3,21,21,76,6,...,0.460000,3,13,6.55350,195.89,804.11,197.66,1.64,5.16,1.235528
9,1078.143,902.096,0.253165,74,33,7,24,20,79,6,...,0.460000,3,12,7.80750,186.66,813.34,217.11,0.00,5.63,0.113943


In [231]:
dataset1.describe()

,Molecular Weight (MW),CharVol (characteristic volume),Flexibility (number of rotatable bonds / number of bonds),Number of Heavy Atoms (HA),RingAtoms,Halogens,HeteroAtoms,RotBonds (NRotB),AllBonds,RingCount,...,Fraction of sp3 Carbon Atoms (FSP3),Hydrogen Bond Donors (HBD),Hydrogen Bond Acceptors (HBA),cLogD^7.4,Topological polar surface area (TPSA),Total non-polar surface area (TNSA),Ensemble_Average_PSA_Chloroform_ANI,Ensemble_Average_Num_IMHB_Chloroform_ANI,Ensemble_Average_RadiusOfGyration_Chloroform_ANI,P_appLog
count,32.000000,32.000000,32.000000,32.000000,32.000000,32.000000,32.000000,32.000000,32.000000,32.000000,...,32.000000,32.000000,32.000000,32.000000,32.000000,32.000000,32.000000,32.000000,32.000000,32.000000
mean,977.812031,851.780000,0.248764,68.468750,40.906250,2.062500,19.906250,18.718750,74.718750,7.250000,...,0.432688,3.500000,13.531250,6.182783,204.767500,766.482500,183.080625,1.950625,5.650937,0.761987
std,134.779959,118.170673,0.047033,8.511792,5.855019,1.933366,2.809654,4.502575,8.865425,0.983739,...,0.058824,1.107161,1.917397,2.005894,25.445808,122.333173,47.226018,1.229854,0.883882,0.612052
min,722.847000,661.968000,0.092308,53.000000,33.000000,0.000000,13.000000,6.000000,58.000000,6.000000,...,0.209302,1.000000,9.000000,1.763900,137.370000,602.610000,105.630000,0.000000,4.640000,-1.000000
25%,895.363750,768.226000,0.233296,62.000000,39.000000,1.000000,18.750000,16.000000,68.750000,7.000000,...,0.418261,3.000000,12.000000,5.177125,186.660000,707.920000,144.055000,0.915000,5.247500,0.524897
50%,1004.422000,876.584000,0.259299,69.000000,40.500000,2.000000,20.000000,20.000000,75.000000,7.000000,...,0.449495,3.000000,13.500000,6.572750,208.505000,771.875000,174.995000,2.020000,5.415000,0.844699
75%,1038.975000,903.506000,0.276316,73.000000,44.000000,3.000000,21.250000,21.000000,79.250000,8.000000,...,0.460741,4.000000,15.000000,7.411925,217.692500,812.117500,207.405000,2.940000,5.657500,1.139845
max,1459.411000,1291.704000,0.337500,101.000000,58.000000,9.000000,26.000000,27.000000,110.000000,10.000000,...,0.540000,6.000000,17.000000,12.293000,262.760000,1303.210000,301.960000,3.970000,8.770000,1.690196


In [232]:
#A

In [233]:
import numpy as np

# ---- 1) Find the log-permeability column automatically ----
def find_col_contains(df, must_contain_any):
    must_contain_any = [s.lower() for s in must_contain_any]
    for c in df.columns:
        cl = c.lower()
        if any(s in cl for s in must_contain_any):
            return c
    return None

# Try to find a "P_appLog" / "log" permeability column
papp_log_col = None

# (A) exact/common names first
for cand in ["P_appLog", "PappLog", "P_app_log", "Papp_log", "logP_app", "logPapp", "papp_log"]:
    if cand in dataset.columns:
        papp_log_col = cand
        break

# (B) substring search fallback
if papp_log_col is None:
    # looks for columns containing both "p" and "log" and "app" (in any form)
    # common patterns: "papp", "p_app", "app", plus "log"
    papp_log_col = find_col_contains(dataset, ["log"])  # start with any log column
    # If there are multiple log columns in your dataset, refine:
    log_cols = [c for c in dataset.columns if "log" in c.lower()]
    # Prefer those that also mention papp/app/permeab
    preferred = [c for c in log_cols if ("papp" in c.lower() or "p_app" in c.lower() or "app" in c.lower() or "permeab" in c.lower())]
    if preferred:
        papp_log_col = preferred[0]

if papp_log_col is None:
    raise KeyError(
        "I couldn't find the log permeability column (e.g., P_appLog). "
        "Run: print(dataset.columns) and check what the log-Papp column is called."
    )

print("Using log-permeability column:", papp_log_col)

# ---- 2) Convert log10(P_app) back to P_app (nm/s) ----
dataset["P_app (nm/s)"] = 10 ** pd.to_numeric(dataset[papp_log_col], errors="coerce")

# ---- 3) Median cutoff and Low/High labels ----
Permeable_cutoff = dataset["P_app (nm/s)"].median()
Low_label = 0
High_label = 1

Permeable_key_str = f"Permeability high({High_label})/Low({Low_label})"
dataset[Permeable_key_str] = np.where(dataset["P_app (nm/s)"] >= Permeable_cutoff, High_label, Low_label)

Number_Permeable_High = int((dataset[Permeable_key_str] == High_label).sum())
Number_Permeable_Low  = int((dataset[Permeable_key_str] == Low_label).sum())

print("Key:", Permeable_key_str)
print(f"Number with high permeability (>= {Permeable_cutoff:.2f} nm/s): {Number_Permeable_High}")
print(f"Number with low  permeability (<  {Permeable_cutoff:.2f} nm/s): {Number_Permeable_Low}")

dataset[["P_app (nm/s)", Permeable_key_str]].head(10)

Using log-permeability column: Halogens
Key: Permeability high(1)/Low(0)
Number with high permeability (>= 100.00 nm/s): 17
Number with low  permeability (<  100.00 nm/s): 15


,P_app (nm/s),Permeability high(1)/Low(0)
0,1.000000e+00,0
1,1.000000e+00,0
2,1.000000e+00,0
3,1.000000e+03,1
4,1.000000e+01,0
5,1.000000e+03,1
6,1.000000e+03,1
7,1.000000e+09,1
8,1.000000e+03,1
9,1.000000e+07,1


In [234]:
print(dataset.keys())

Index(['Index', 'Compound', 'Smiles', 'P_app AB (nm/s)', 'P_app BA (nm/s)',
       'P_app', 'Molecular Weight (MW)', 'CharVol (characteristic volume)',
       'Flexibility (number of rotatable bonds / number of bonds)',
       'Number of Heavy Atoms (HA)', 'RingAtoms', 'Halogens', 'HeteroAtoms',
       'RotBonds (NRotB)', 'AllBonds', 'RingCount', 'NumStereo',
       'Fraction of sp3 Carbon Atoms (FSP3)', 'Hydrogen Bond Donors (HBD)',
       'Hydrogen Bond Acceptors (HBA)', 'cLogD^7.4',
       'Topological polar surface area (TPSA)',
       'Total non-polar surface area (TNSA)', 'P_app (nm/s)',
       'Permeability high(1)/Low(0)'],
      dtype='object')


In [235]:
#B

In [236]:
import pandas as pd
import numpy as np

# dataset_1 = your dataframe with 32 compounds
# Make sure dataset_1 exists before running this cell.

# --- auto-find the permeability column (log or linear) ---
def find_first_col(df, candidates, contains_any=()):
    # exact match first
    for c in candidates:
        if c in df.columns:
            return c
    # case-insensitive exact match
    lower_map = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    # substring match
    for col in df.columns:
        col_l = col.lower()
        if any(tok in col_l for tok in contains_any):
            return col
    return None

# If you already have P_app in nm/s:
papp_nm_col = find_first_col(dataset_1,
                            candidates=["P_app(nm/s)", "P_app (nm/s)", "P_app_nm_s", "Papp(nm/s)", "Papp_nm_s"],
                            contains_any=("nm/s", "nm_s"))

# If you only have log(P_app):
papp_log_col = find_first_col(dataset_1,
                             candidates=["P_appLog", "PappLog", "P_app_log", "logPapp", "logP_app"],
                             contains_any=("log", "papp", "p_app", "permeab"))

# Convert to nm/s if needed
if papp_nm_col is None:
    if papp_log_col is None:
        raise KeyError("Couldn't find permeability column. Try: print(dataset_1.columns.tolist())")
    dataset_1["P_app(nm/s)"] = (10 ** pd.to_numeric(dataset_1[papp_log_col], errors="coerce"))
else:
    dataset_1["P_app(nm/s)"] = pd.to_numeric(dataset_1[papp_nm_col], errors="coerce")

# Round like your screenshot
dataset_1["P_app(nm/s)"] = dataset_1["P_app(nm/s)"].round(2)

# Median cutoff (same as your approach)
Permeable_cutoff = dataset_1["P_app(nm/s)"].median()
Low_label = 0
High_label = 1

Permeable_key_str = f"Permeability high({High_label})/Low({Low_label})"
dataset_1[Permeable_key_str] = np.where(dataset_1["P_app(nm/s)"] >= Permeable_cutoff, High_label, Low_label)

print("Key:", Permeable_key_str)
print(f"Cutoff (median) = {Permeable_cutoff:.2f} nm/s")

Key: Permeability high(1)/Low(0)
Cutoff (median) = 7.00 nm/s


In [237]:
#C

In [238]:
print(dataset.columns.tolist())

['Index', 'Compound', 'Smiles', 'P_app AB (nm/s)', 'P_app BA (nm/s)', 'P_app', 'Molecular Weight (MW)', 'CharVol (characteristic volume)', 'Flexibility (number of rotatable bonds / number of bonds)', 'Number of Heavy Atoms (HA)', 'RingAtoms', 'Halogens', 'HeteroAtoms', 'RotBonds (NRotB)', 'AllBonds', 'RingCount', 'NumStereo', 'Fraction of sp3 Carbon Atoms (FSP3)', 'Hydrogen Bond Donors (HBD)', 'Hydrogen Bond Acceptors (HBA)', 'cLogD^7.4', 'Topological polar surface area (TPSA)', 'Total non-polar surface area (TNSA)', 'P_app (nm/s)', 'Permeability high(1)/Low(0)']


In [239]:

print("Target value counts:")
print(target.value_counts())

print("\nPositive class (High=1):", np.sum(target == 1))
print("Negative class (Low=0):", np.sum(target == 0))

Target value counts:
Permeability High(1)/Low(0)
1    16
0    16
Name: count, dtype: int64

Positive class (High=1): 16
Negative class (Low=0): 16


In [240]:
import math
from sklearn.ensemble import RandomForestClassifier

# Calculate max_features for each feature set (50% rounded up)
max_features_2D = math.ceil(features_2D.shape[1] * 0.5)
max_features_3D = math.ceil(features_3D.shape[1] * 0.5)
max_features_combined = math.ceil(features_combined.shape[1] * 0.5)

print("Number of 2D features:", features_2D.shape[1], "-> max_features:", max_features_2D)
print("Number of 3D features:", features_3D.shape[1], "-> max_features:", max_features_3D)
print("Number of Combined features:", features_combined.shape[1], "-> max_features:", max_features_combined)

rf_2D = RandomForestClassifier(n_estimators=100, max_depth=2, max_features=max_features_2D, random_state=42)
rf_3D = RandomForestClassifier(n_estimators=100, max_depth=2, max_features=max_features_3D, random_state=42)
rf_combined = RandomForestClassifier(n_estimators=100, max_depth=2, max_features=max_features_combined, random_state=42)

print("Models defined successfully!")

Number of 2D features: 17 -> max_features: 9
Number of 3D features: 3 -> max_features: 2
Number of Combined features: 20 -> max_features: 10
Models defined successfully!


In [241]:
from sklearn.model_selection import StratifiedKFold

# Use Stratified 5-Fold CV to ensure equal class balance in each fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Cross Validation strategy: Stratified 5-Fold")
print("Number of splits:", cv.n_splits)

Cross Validation strategy: Stratified 5-Fold
Number of splits: 5


In [242]:
from sklearn.model_selection import cross_val_score
import numpy as np

def evaluate_model(model, features, target, cv):
    accuracy = cross_val_score(model, features, target, cv=cv, scoring='accuracy')
    precision = cross_val_score(model, features, target, cv=cv, scoring='precision')
    roc_auc = cross_val_score(model, features, target, cv=cv, scoring='roc_auc')

    print(f"  Accuracy:  {accuracy.mean():.3f} +/- {accuracy.std():.3f}")
    print(f"  Precision: {precision.mean():.3f} +/- {precision.std():.3f}")
    print(f"  ROC AUC:   {roc_auc.mean():.3f} +/- {roc_auc.std():.3f}")

    return accuracy.mean(), precision.mean(), roc_auc.mean()

print("=== 2D RDKit Features ===")
acc_2D, prec_2D, auc_2D = evaluate_model(rf_2D, features_2D, target, cv)

print("\n=== 3D Ensemble Features ===")
acc_3D, prec_3D, auc_3D = evaluate_model(rf_3D, features_3D, target, cv)

print("\n=== Combined 2D + 3D Features ===")
acc_comb, prec_comb, auc_comb = evaluate_model(rf_combined, features_combined, target, cv)

print("\n=== Summary Table ===")
print(f"{'Feature Set':<20} {'Accuracy':>10} {'Precision':>10} {'ROC AUC':>10}")
print("-" * 52)
print(f"{'2D RDKit':<20} {acc_2D:>10.3f} {prec_2D:>10.3f} {auc_2D:>10.3f}")
print(f"{'3D Ensemble':<20} {acc_3D:>10.3f} {prec_3D:>10.3f} {auc_3D:>10.3f}")
print(f"{'Combined':<20} {acc_comb:>10.3f} {prec_comb:>10.3f} {auc_comb:>10.3f}")

=== 2D RDKit Features ===
  Accuracy:  0.386 +/- 0.181
  Precision: 0.350 +/- 0.374
  ROC AUC:   0.294 +/- 0.216

=== 3D Ensemble Features ===
  Accuracy:  0.790 +/- 0.142
  Precision: 0.803 +/- 0.167
  ROC AUC:   0.750 +/- 0.247

=== Combined 2D + 3D Features ===
  Accuracy:  0.671 +/- 0.198
  Precision: 0.730 +/- 0.248
  ROC AUC:   0.717 +/- 0.258

=== Summary Table ===
Feature Set            Accuracy  Precision    ROC AUC
----------------------------------------------------
2D RDKit                  0.386      0.350      0.294
3D Ensemble               0.790      0.803      0.750
Combined                  0.671      0.730      0.717


In [243]:
print("""
=== Conclusion ===

The 3D Ensemble features give the best classification results across all metrics:

  - Accuracy:  0.790 (vs 0.386 for 2D, 0.671 for Combined)
  - Precision: 0.803 (vs 0.350 for 2D, 0.730 for Combined)
  - ROC AUC:   0.750 (vs 0.294 for 2D, 0.717 for Combined)

The 2D RDKit features perform the worst, with accuracy (0.386) and ROC AUC (0.294)
even below random chance (0.5), suggesting these features alone are not sufficient
to predict permeability.

The Combined (2D + 3D) features perform better than 2D alone but worse than 3D alone,
suggesting that adding 2D features to the 3D features introduces noise that slightly
hurts the model performance.

Overall, the 3D Ensemble features derived from molecular dynamics simulations
(PSA, IMHB, Radius of Gyration) are the most informative for predicting
PROTAC permeability with this Random Forest model.
""")


=== Conclusion ===

The 3D Ensemble features give the best classification results across all metrics:

  - Accuracy:  0.790 (vs 0.386 for 2D, 0.671 for Combined)
  - Precision: 0.803 (vs 0.350 for 2D, 0.730 for Combined)
  - ROC AUC:   0.750 (vs 0.294 for 2D, 0.717 for Combined)

The 2D RDKit features perform the worst, with accuracy (0.386) and ROC AUC (0.294)
even below random chance (0.5), suggesting these features alone are not sufficient
to predict permeability.

The Combined (2D + 3D) features perform better than 2D alone but worse than 3D alone,
suggesting that adding 2D features to the 3D features introduces noise that slightly
hurts the model performance.

Overall, the 3D Ensemble features derived from molecular dynamics simulations
(PSA, IMHB, Radius of Gyration) are the most informative for predicting
PROTAC permeability with this Random Forest model.



In [244]:
#D

In [245]:
from sklearn.neighbors import KNeighborsClassifier

# Define KNN models for each feature set (k=5 default)
knn_2D = KNeighborsClassifier(n_neighbors=5)
knn_3D = KNeighborsClassifier(n_neighbors=5)
knn_combined = KNeighborsClassifier(n_neighbors=5)

print("=== KNN (k=5) Results ===\n")

print("=== 2D RDKit Features ===")
acc_knn_2D, prec_knn_2D, auc_knn_2D = evaluate_model(knn_2D, features_2D, target, cv)

print("\n=== 3D Ensemble Features ===")
acc_knn_3D, prec_knn_3D, auc_knn_3D = evaluate_model(knn_3D, features_3D, target, cv)

print("\n=== Combined 2D + 3D Features ===")
acc_knn_comb, prec_knn_comb, auc_knn_comb = evaluate_model(knn_combined, features_combined, target, cv)

print("\n=== KNN Summary Table ===")
print(f"{'Feature Set':<20} {'Accuracy':>10} {'Precision':>10} {'ROC AUC':>10}")
print("-" * 52)
print(f"{'2D RDKit':<20} {acc_knn_2D:>10.3f} {prec_knn_2D:>10.3f} {auc_knn_2D:>10.3f}")
print(f"{'3D Ensemble':<20} {acc_knn_3D:>10.3f} {prec_knn_3D:>10.3f} {auc_knn_3D:>10.3f}")
print(f"{'Combined':<20} {acc_knn_comb:>10.3f} {prec_knn_comb:>10.3f} {auc_knn_comb:>10.3f}")

print("\n=== Comparison: Random Forest vs KNN ===")
print(f"{'Feature Set':<20} {'RF Accuracy':>12} {'KNN Accuracy':>13}")
print("-" * 47)
print(f"{'2D RDKit':<20} {acc_2D:>12.3f} {acc_knn_2D:>13.3f}")
print(f"{'3D Ensemble':<20} {acc_3D:>12.3f} {acc_knn_3D:>13.3f}")
print(f"{'Combined':<20} {acc_comb:>12.3f} {acc_knn_comb:>13.3f}")

=== KNN (k=5) Results ===

=== 2D RDKit Features ===
  Accuracy:  0.505 +/- 0.309
  Precision: 0.457 +/- 0.366
  ROC AUC:   0.475 +/- 0.291

=== 3D Ensemble Features ===
  Accuracy:  0.500 +/- 0.188
  Precision: 0.497 +/- 0.177
  ROC AUC:   0.464 +/- 0.123

=== Combined 2D + 3D Features ===
  Accuracy:  0.567 +/- 0.176
  Precision: 0.563 +/- 0.165
  ROC AUC:   0.539 +/- 0.199

=== KNN Summary Table ===
Feature Set            Accuracy  Precision    ROC AUC
----------------------------------------------------
2D RDKit                  0.505      0.457      0.475
3D Ensemble               0.500      0.497      0.464
Combined                  0.567      0.563      0.539

=== Comparison: Random Forest vs KNN ===
Feature Set           RF Accuracy  KNN Accuracy
-----------------------------------------------
2D RDKit                    0.386         0.505
3D Ensemble                 0.790         0.500
Combined                    0.671         0.567
